In [ ]:
import copy
import os

import numpy as np
from barrier3d import Barrier3d
from matplotlib import pyplot as plt

from cascade.outwasher import Outwasher

In [ ]:
b3d = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
# years = np.load("C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years.npy")

for t in range(1, 64):
    b3d.update()
    b3d.update_dune_domain()
    print("B3D time step: ", b3d._time_index)

In [ ]:
dunes = b3d.DuneDomain[b3d._time_index - 1]
dunes = np.transpose(dunes) + b3d._BermEl

In [ ]:
domain = b3d.InteriorDomain
full_domain = np.append(dunes, domain, 0)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})


### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
mat = ax1.matshow(
    #     np.flip(full_domain, 0)*10,
    full_domain * 10,
    cmap="terrain",
    #     vmin=-3.0, vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()

In [ ]:
# dunes closest to ocean
dune_array = np.zeros([b3d._time_index - 1, b3d._BarrierLength])
for t in range(1, 64):
    beach_dunes = b3d.DuneDomain[t]
    beach_dunes = np.transpose(beach_dunes) + b3d._BermEl
    beach_dunes = beach_dunes[0]
    dune_array[t - 1] = beach_dunes
# print(dune_array)
fig = plt.figure()
ax = fig.add_subplot(111)
mat = ax.matshow(
    #     np.flip(full_domain, 0)*10,
    dune_array * 10,
    cmap="terrain",
    vmin=-0.5,
    vmax=3.0,
)
cbar = fig.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax.set_title("Dune Elevation closest to Ocean")
ax.set_ylabel("time")
ax.set_yticks([0, 9, 19, 29, 39, 49, 59])
labels = ["1", "10", "20", "30", "40", "50", "60"]
ax.set_yticklabels(labels)
ax.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()

# dunes closest to bay
dune_array = np.zeros([b3d._time_index - 1, b3d._BarrierLength])
for t in range(1, 64):
    beach_dunes = b3d.DuneDomain[t]
    beach_dunes = np.transpose(beach_dunes) + b3d._BermEl
    beach_dunes = beach_dunes[1]
    dune_array[t - 1] = beach_dunes
# print(dune_array)
fig = plt.figure()
ax = fig.add_subplot(111)
mat = ax.matshow(
    #     np.flip(full_domain, 0)*10,
    dune_array * 10,
    cmap="terrain",
    vmin=-0.5,
    vmax=3.0,
)
cbar = fig.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax.set_title("Dune Elevation closest to Bay")
ax.set_ylabel("time")
ax.set_yticks([0, 9, 19, 29, 39, 49, 59])
labels = ["1", "10", "20", "30", "40", "50", "60"]
ax.set_yticklabels(labels)
ax.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()

In [ ]:
b3d2 = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
# years = np.load("C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years.npy")

for t in range(1, 64):
    print("B3D time step: ", b3d2._time_index)
    b3d2.update()
    b3d2.update_dune_domain()
outwash = Outwasher(
    datadir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/",
    outwash_years="outwash_years.npy",
    outwash_bay_levels="outwash_baylevels10.npy",
    time_step_count=b3d2._TMAX,
    berm_elev=b3d2._BermEl,
    barrier_length=b3d2._BarrierLength,
    sea_level=b3d2._SL,
    bay_depth=-b3d2._BayDepth,
    interior_domain=b3d2.InteriorDomain,
    dune_domain=b3d.DuneDomain[b3d2._time_index - 1],
    block_size=5,
    substep=20,
    sediment_flux_coefficient_Cx=10,
    sediment_flux_coefficient_Ki=7.5e-3,  # b3d = 7.5E-6 for inundation
    max_slope=-0.25,
    shoreface_on=True,
)
outwash.update(b3d2)

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

initial_domain = outwash._initial_full_domain
final_domain = outwash._full_domain
domain_change = final_domain - initial_domain

### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
mat = ax1.matshow(
    initial_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()

# plotting post storm elevation
fig2 = plt.figure()
ax2 = fig2.add_subplot(111)
mat2 = ax2.matshow(
    final_domain[:, :] * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
ax2.set_xlabel("barrier length (dam)")
ax2.set_ylabel("barrier width (dam)")
ax2.set_title("Final Elevation")
plt.gca().xaxis.tick_bottom()
cbar = fig2.colorbar(mat2)
cbar.set_label("m MHW", rotation=270, labelpad=15)

# plotting domain elevation change
domain_change = final_domain - initial_domain
fig3 = plt.figure()
ax3 = fig3.add_subplot(111)
mat3 = ax3.matshow(
    domain_change * 10,
    cmap="seismic",
    vmin=-3,
    vmax=3,
)
ax3.set_xlabel("barrier length (dam)")
ax3.set_ylabel("barrier width (dam)")
ax3.set_title("Elevation Change")
plt.gca().xaxis.tick_bottom()
cbar = fig3.colorbar(mat3)
cbar.set_label("meters", rotation=270, labelpad=15)

In [ ]:
# dunes closest to beach

dune_array2 = np.zeros([b3d2._time_index - 1, b3d2._BarrierLength])
for t in range(1, 64):
    beach_dunes2 = b3d2.DuneDomain[t]
    beach_dunes2 = np.transpose(beach_dunes2) + b3d2._BermEl
    beach_dunes2 = beach_dunes2[0]
    dune_array2[t - 1] = beach_dunes2
# print(dune_array)
fig = plt.figure()
ax = fig.add_subplot(111)
mat = ax.matshow(
    #     np.flip(full_domain, 0)*10,
    dune_array2 * 10,
    cmap="terrain",
    vmin=-0.5,
    vmax=3.0,
)
cbar = fig.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax.set_title("Dune Elevation closest to Ocean")
ax.set_ylabel("time")
ax.set_yticks([0, 9, 19, 29, 39, 49, 59])
labels = ["1", "10", "20", "30", "40", "50", "60"]
ax.set_yticklabels(labels)
ax.set_xlabel("barrier length (dam)")
# xticklabels({'1, 10, 20, 30, 40, 40, 60'})
plt.gca().xaxis.tick_bottom()

# dunes closest to beach

dune_array2 = np.zeros([b3d2._time_index - 1, b3d2._BarrierLength])
for t in range(1, 64):
    beach_dunes2 = b3d2.DuneDomain[t]
    beach_dunes2 = np.transpose(beach_dunes2) + b3d2._BermEl
    beach_dunes2 = beach_dunes2[1]
    dune_array2[t - 1] = beach_dunes2
# print(dune_array)
fig = plt.figure()
ax = fig.add_subplot(111)
mat = ax.matshow(
    #     np.flip(full_domain, 0)*10,
    dune_array2 * 10,
    cmap="terrain",
    vmin=-0.5,
    vmax=3.0,
)
cbar = fig.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax.set_title("Dune Elevation closest to Bay")
ax.set_ylabel("time")
ax.set_yticks([0, 9, 19, 29, 39, 49, 59])
labels = ["1", "10", "20", "30", "40", "50", "60"]
ax.set_yticklabels(labels)
ax.set_ylim(bottom=None, top=1)
ax.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()

In [ ]:
dunes = b3d2.DuneDomain[b3d2._time_index - 1]
dunes = np.transpose(dunes) + b3d2._BermEl
domain = b3d2.InteriorDomain
full_domain = np.append(dunes, domain, 0)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})


### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig4 = plt.figure()
ax4 = fig4.add_subplot(111)
mat4 = ax4.matshow(
    #     np.flip(full_domain, 0)*10,
    full_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig4.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax4.set_title("Initial Elevation")
ax4.set_ylabel("barrier width (dam)")
ax4.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()

In [ ]:
plt.rcParams["figure.figsize"] = (10, 10)
plt.rcParams.update({"font.size": 15})
dune_array2 = np.zeros([100, b3d2._BarrierLength])
for t in range(1, 101):
    beach_dunes2 = b3d2.DuneDomain[t]
    beach_dunes2 = np.transpose(beach_dunes2) + b3d2._BermEl
    beach_dunes2 = beach_dunes2[0]
    dune_array2[t - 1] = beach_dunes2
# print(dune_array)
fig = plt.figure()
ax = fig.add_subplot(111)
mat = ax.matshow(
    #     np.flip(full_domain, 0)*10,
    dune_array2 * 10,
    cmap="terrain",
    vmin=-0.5,
    vmax=3.0,
)
cbar = fig.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax.set_title("Dune Elevation closest to Ocean")
ax.set_ylabel("time")
ax.set_yticks([0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99])
labels = ["1", "10", "20", "30", "40", "50", "60", "70", "80", "90", "100"]
ax.set_yticklabels(labels)
ax.set_xlabel("barrier length (dam)")
# xticklabels({'1, 10, 20, 30, 40, 40, 60'})
plt.gca().xaxis.tick_bottom()